In [2]:
import pandas as pd

# Cargar el dataset
df = pd.read_csv('M08.csv')

# 1. Ver la cantidad total de filas y las columnas
print("Dimensiones:", df.shape)
print("\nColumnas y valores nulos:")
print(df.info())

# 2. Ver las primeras filas para entender qué datos tenemos
print("\nPrimeras filas:")
display(df.head())

# 3. Ver el balance de la clase a predecir.
print("\nBalance de clases:")
print(df['resultado_real'].value_counts())

Dimensiones: (20, 7)

Columnas y valores nulos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   caso                                    20 non-null     int64  
 1   conjunto                                20 non-null     object 
 2   Fondo más largo (km)                    20 non-null     int64  
 3   Ritmo en 10K (min por km)               20 non-null     float64
 4   Semanas sin lesión (de las últimas 12)  20 non-null     int64  
 5   Km semanales (km)                       20 non-null     int64  
 6   resultado_real                          20 non-null     int64  
dtypes: float64(1), int64(5), object(1)
memory usage: 1.2+ KB
None

Primeras filas:


,caso,conjunto,Fondo más largo (km),Ritmo en 10K (min por km),Semanas sin lesión (de las últimas 12),Km semanales (km),resultado_real
0,1,entrenamiento,29,6.6,10,72,0
1,2,entrenamiento,13,6.5,8,61,0
2,3,entrenamiento,31,4.3,5,25,1
3,4,entrenamiento,27,6.2,7,45,0
4,5,entrenamiento,33,6.4,12,78,1



Balance de clases:
resultado_real
0    10
1    10
Name: count, dtype: int64


In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Separar según la columna 'conjunto'
df_train = df[df['conjunto'] == 'entrenamiento']
df_test = df[df['conjunto'] == 'prueba']

print(f"Filas de entrenamiento: {len(df_train)}")
print(f"Filas de prueba: {len(df_test)}")

# 2. Definir las variables predictoras (X) y el objetivo (y)
# ELIMINAMOS 'caso' (es un ID) y 'conjunto' (ya la usamos). 'resultado_real' es lo que predecimos.
columnas_x = ['Fondo más largo (km)', 'Ritmo en 10K (min por km)', 'Semanas sin lesión (de las últimas 12)', 'Km semanales (km)']

X_train = df_train[columnas_x]
y_train = df_train['resultado_real']

X_test = df_test[columnas_x]
y_test = df_test['resultado_real']

# 3. Entrenar el modelo (Árbol de Decisión simple)
# Usamos un max_depth bajo para evitar que se sobreajuste (overfitting) a tan pocos datos
modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_train, y_train)

# 4. Hacer predicciones
y_pred_train = modelo.predict(X_train)
y_pred_test = modelo.predict(X_test)

# 5. Ver matrices y métricas de PRUEBA
cm_test = confusion_matrix(y_test, y_pred_test)
print("\n--- MÉTRICAS EN PRUEBA ---")
print("Matriz de Confusión (Test):")
print(cm_test)
print(f"Exactitud (Accuracy): {accuracy_score(y_test, y_pred_test):.2f}")
print(f"Precisión: {precision_score(y_test, y_pred_test, zero_division=0):.2f}")
print(f"Sensibilidad (Recall): {recall_score(y_test, y_pred_test, zero_division=0):.2f}")

Filas de entrenamiento: 14
Filas de prueba: 6

--- MÉTRICAS EN PRUEBA ---
Matriz de Confusión (Test):
[[3 0]
 [2 1]]
Exactitud (Accuracy): 0.67
Precisión: 1.00
Sensibilidad (Recall): 0.33


In [4]:
print("\n--- MÉTRICAS EN ENTRENAMIENTO ---")
cm_train = confusion_matrix(y_train, y_pred_train)
print("Matriz de Confusión (Train):")
print(cm_train)
print(f"Exactitud (Accuracy): {accuracy_score(y_train, y_pred_train):.2f}")
print(f"Precisión: {precision_score(y_train, y_pred_train, zero_division=0):.2f}")
print(f"Sensibilidad (Recall): {recall_score(y_train, y_pred_train, zero_division=0):.2f}")


--- MÉTRICAS EN ENTRENAMIENTO ---
Matriz de Confusión (Train):
[[7 0]
 [0 7]]
Exactitud (Accuracy): 1.00
Precisión: 1.00
Sensibilidad (Recall): 1.00


In [5]:
from sklearn.tree import export_text

# Extraer y mostrar las reglas lógicas del árbol
reglas = export_text(modelo, feature_names=columnas_x)
print("REGLAS DEL MODELO:")
print(reglas)

REGLAS DEL MODELO:
|--- Ritmo en 10K (min por km) <= 5.60
|   |--- Fondo más largo (km) <= 21.00
|   |   |--- Km semanales (km) <= 49.50
|   |   |   |--- class: 1
|   |   |--- Km semanales (km) >  49.50
|   |   |   |--- class: 0
|   |--- Fondo más largo (km) >  21.00
|   |   |--- class: 1
|--- Ritmo en 10K (min por km) >  5.60
|   |--- Fondo más largo (km) <= 32.00
|   |   |--- class: 0
|   |--- Fondo más largo (km) >  32.00
|   |   |--- class: 1

